Zaimportowanie bibliotek

In [1]:
import numpy as np
import math
from scipy.optimize import brentq # do wyliczenia d max
from scipy.special import erfc # do BER przy różnych modulacjach

Wybór scenariusza komunikacji

In [2]:
scenariusz = input("Choose the communication scenario — ground one (N) or underwater one (P): ").strip().upper()

if scenariusz == "N":
    print("Scenario - ground communication - free space optical comm (FSO)")
elif scenariusz == "P":
    print("Scenario - underwater comm (UWOC)")
else:
    print("Error - choose ground (N) or underwater (P) communication to make calculations.")

Error - choose ground (N) or underwater (P) communication to make calculations.


Wybór modulacji

In [3]:
modulation = input("Choose modulation format (OOK/PPM/PAM/BPSK): ").strip().upper()

M_order = 1

if modulation == "PPM":
    M_order = int(input("Enter M for M-PPM (e.g. 2, 4, 8, 16): "))
    print(f"Selected: {M_order} PPM")
elif modulation == "PAM":
    M_order = int(input("Enter M for M-PAM (e.g. 2, 4, 8): "))
    print(f"Selected: {M_order} PAM")
elif modulation == "OOK":
    print("Selected: OOK")
elif modulation == "BPSK":
    print("Selected: BPSK")
else:
    modulation = "OOK"
    print("Unknown modulation - assuming the OOK model")

Unknown modulation - assuming the OOK model


Funkcje BER dla różnych modulacji

In [4]:
def BER_OOK(SNR):
    return 0.5 * erfc(math.sqrt(SNR / 2))

def BER_BPSK(SNR):
    return 0.5 * erfc(math.sqrt(SNR))

def BER_PPM(SNR, M):
    return ((M - 1) / (2 * M * math.log2(M))) * erfc(math.sqrt(SNR / (M - 1)))

def BER_PAM(SNR, M):
    return 0.5 * erfc(math.sqrt(SNR * math.log2(M)) / (2 * math.sqrt(2) * (M - 1)))

Wzory związane ze stratami - komunikacja podwodna (UWOC)

In [5]:
def L_water():
    water_condition = input("Enter the water condition: clear ocean (CO), coastal water (CSW) or turbid harbor (TH): ").strip().upper()

    if water_condition == "CO":
        c_factor = 0.15
    elif water_condition == "CSW":
        c_factor = 0.30
    elif water_condition == "TH":
        c_factor = 0.70
    else:
        print("Unknown water type, assuming: clear ocean (c=0.15)")
        c_factor = 0.15

    return c_factor

In [6]:
def P_received_water(P_T_W, theta_rad, d_m, D_R_m, c_factor):
    eta_geo   = (D_R_m / (theta_rad * d_m))**2
    attenuation = math.exp(-c_factor * d_m)
    P_R_W = P_T_W * eta_geo * attenuation
    return P_R_W

Wzory związane ze stratami - komunikacja naziemna (FSO)

In [7]:
def L_geometric(beam_divergence_rad, receiver_diameter_m, link_distance_m):
    L_geo = 20 * math.log10((beam_divergence_rad * link_distance_m) / receiver_diameter_m)
    return L_geo

In [8]:
def L_atmospheric(visibility_km, wavelength_um, link_distance_km):
    # model Kruse'a
    if visibility_km > 50:
        q_factor = 1.6
    elif visibility_km > 6:
        q_factor = 1.3
    elif visibility_km > 1:
        q_factor = 0.585 * visibility_km**(1/3)
    else:
        q_factor = 0

    attenuation = (3.91 / visibility_km) * (wavelength_um / 0.55)**(-q_factor)
    L_atm = attenuation * link_distance_km
    return L_atm

In [9]:
def L_total_earth(L_geo, L_atm):
    L_others_str = input("If there are any other losses (pointing, optics), enter their value (in dB). If none, enter 'NO': ").strip()

    if L_others_str.upper() == 'NO':
        L_extra = 0
    else:
        L_extra = float(L_others_str)

    L_tot = L_geo + L_atm + L_extra
    return L_tot, L_extra

Scyntylacja atmosferyczna (dla FSO) - model Rytova

In [10]:
def L_scintillation(wavelength_m, link_distance_m, Cn2):
    k = 2 * math.pi / wavelength_m
    sigma2_chi = 23.17 * (k**(7/6)) * Cn2 * (link_distance_m**(11/6))
    A_fade = 2 * math.sqrt(sigma2_chi)
    return sigma2_chi, A_fade

Ustalenie wartości mocy nadajnika (+ przeliczenie na dBm)

In [11]:
P_transmitted = float(input("Enter the transmitted power value: "))
PTunit = input("Is the power given in dBm (enter dBm) or mW (enter mW)?: ").strip()

if PTunit == "mW":
    P_transmitted_dBm = 10 * math.log10(P_transmitted)
    print(f"Transmitted power: {P_transmitted} mW = {P_transmitted_dBm:.2f} dBm")
else:
    P_transmitted_dBm = P_transmitted
    print(f"Transmitted power: {P_transmitted_dBm:.2f} dBm")

ValueError: could not convert string to float: ''

Ustalenie minimalnej mocy odbieranej (P min) - czułość detektora

In [ ]:
P_min = float(input("Enter the minimum detectable power (detector sensitivity): "))
Pminunit = input("Is the power given in dBm (enter dBm) or mW (enter mW)?: ").strip()

if Pminunit == "mW":
    P_min_dBm = 10 * math.log10(P_min)
    print(f"Minimum detectable power: {P_min} mW = {P_min_dBm:.2f} dBm")
else:
    P_min_dBm = P_min
    print(f"Minimum detectable power: {P_min_dBm:.2f} dBm")

Minimum detectable power: 1.00 dBm


Obliczenie mocy odbieranej i sprawdzenie łącza — komunikacja NAZIEMNA

In [ ]:
if scenariusz == "N":
    beam_divergence_mrad = float(input("Enter the beam divergence (full angle, in miliradians): "))
    receiver_diameter_mm = float(input("Enter the receiver aperture diameter (in milimeters): "))
    link_distance = float(input("Enter the link distance (in meters): "))
    visibility = float(input("Enter the visibility (in kilometers): "))
    wavelength = float(input("Enter the wavelength (in micrometers, e.g. 1.55 for wave 1550 nm): "))

    beam_divergence = beam_divergence_mrad*10e-3
    receiver_diameter = receiver_diameter_mm*10e-3

    L_geo = L_geometric(beam_divergence, receiver_diameter, link_distance)
    L_atm = L_atmospheric(visibility, wavelength, link_distance / 1000)
    L_tot, L_extra = L_total_earth(L_geo, L_atm)

    P_received_dBm = P_transmitted_dBm - L_tot
    P_received_mW = 10**(P_received_dBm / 10)

    print(f"\nLink Budget: Calculated Results")
    print(f"Geometric loss: {L_geo:.2f} dB")
    print(f"Atmospheric loss: {L_atm:.2f} dB")
    print(f"Pointing/other losses: {L_extra:.2f} dB")
    print(f"Total loss: {L_tot:.2f} dB")
    print(f"Received power: {P_received_dBm:.2f} dBm")

    link_margin = P_received_dBm - P_min_dBm
    if link_margin >= 0:
        print(f"Link status: COMM SUCCESSFUL (margin = {link_margin:.2f} dB)")
    else:
        print(f"Link status:  FAIL  (deficit = {abs(link_margin):.2f} dB)") # abs daje wartość bezwzględną


Link Budget: Calculated Results
Geometric loss: 28.98 dB
Atmospheric loss: 0.59 dB
Pointing/other losses: 4.00 dB
Total loss: 33.57 dB
Received power: -23.57 dBm
Link status:  FAIL  (deficit = 24.57 dB)


Scyntylacje - opcjonalnie (tylko dla FSO)

In [ ]:
if scenariusz == "N":
    scint = input("Include scintillation (atmospheric turbulence)? (enter YES or NO): ").strip().upper()

    if scint == "YES":
        print("Typical Cn2 values:")
        print("weak turbulence: 1e-17")
        print("medium turbulence: 1e-15")
        print("strong turbulence: 1e-13")

        Cn2 = float(input("Enter Cn2 (e.g. 1e-15): "))
        wavelength_m = wavelength * 1e-6

        sigma2_chi, A_fade = L_scintillation(wavelength_m, link_distance, Cn2)

        P_received_scint_dBm = P_received_dBm - A_fade
        link_margin_scint = P_received_scint_dBm - P_min_dBm

        print(f"\nScintillation Results")
        print(f"Cn2: {Cn2:.2e} m^(-2/3)")
        print(f"Scintillation variance : {sigma2_chi:.4e} dB^2")
        print(f"Fading loss A_fade:{A_fade:.2f} dB")
        print(f"Received power (with scintillations):{P_received_scint_dBm:.2f} dBm")

        if link_margin_scint >= 0:
            print(f"Link status (with scintillations): COMM SUCCESSFUL (margin = {link_margin_scint:.2f} dB)")
        else:
            print(f"Link status (with scinttillations): FAIL (deficit = {abs(link_margin_scint):.2f} dB)")

Obliczenie mocy odbieranej i sprawdzenie łącza - komunikacja PODWODNA

In [ ]:
if scenariusz == "P":
    beam_divergence = float(input("Enter the beam divergence (full angle, in miliradians): "))
    receiver_diameter = float(input("Enter the receiver aperture diameter (in milimeters): "))
    link_distance = float(input("Enter the link distance (in meters): "))

    c_factor = L_water()

    P_transmitted_W = 10**(P_transmitted_dBm / 10) / 1000

    P_received_W = P_received_water(P_transmitted_W, beam_divergence, link_distance, receiver_diameter, c_factor)
    P_received_mW = P_received_W * 1000
    P_received_dBm = 10 * math.log10(P_received_mW)

    print(f"\nLink Budget Results (underwater)")
    print(f"Attenuation factor: c = {c_factor} m^(-1)")
    print(f"Received power: {P_received_dBm:.2f} dBm  ({P_received_W*1e9:.4f} nW)")

    link_margin = P_received_dBm - P_min_dBm
    if link_margin >= 0:
        print(f"Link status: COMM SUCCESSFUL (margin = {link_margin:.2f} dB)")
    else:
        print(f"Link status: FAIL (deficit = {abs(link_margin):.2f} dB)")

Określanie maksymalnej odległości działania komunikacji (d_max)

In [ ]:
if scenariusz == "N":
    L_budget = P_transmitted_dBm - P_min_dBm

    def equation_to_solve(d):
        L_geo_d = 20 * math.log10((beam_divergence * d) / receiver_diameter)
        L_atm_d = L_atmospheric(visibility, wavelength, d / 1000)
        return L_geo_d + L_atm_d + L_extra - L_budget

    f_at_1m = equation_to_solve(1) # do sprawdzenia, czy podane dane będą się zgadzać i brentq będzie w stanie znaleźć miejsce zerowe do rozwiązania równania na d max
    f_at_500km = equation_to_solve(500_000)

    if f_at_1m * f_at_500km > 0:
        print("WARNING: d_max cannot be found in range 1 m – 500 km.") # założenie, żeby rozpatrywać komunikację w zakresie 1 m - 500 km

    else:
        d_max = brentq(equation_to_solve, 1, 500_000)
        print(f"Maximum comm distance: d_max = {d_max:.1f} m  ({d_max/1000:.3f} km)")

elif scenariusz == "P":
    L_budget_linear = 10**(P_transmitted_dBm / 10) / 1000 / (10**(P_min_dBm / 10) / 1000)

    def equation_to_solve_water(d):
        eta_geo = (receiver_diameter / (beam_divergence * d))**2
        attenuation = math.exp(-c_factor * d)
        return eta_geo * attenuation * L_budget_linear - 1

    f_at_10m = equation_to_solve_water(10) # do sprawdzenia, czy podane dane będą się zgadzać i brentq będzie w stanie znaleźć miejsce zerowe do rozwiązania równania na d max
    f_at_100m = equation_to_solve_water(100)

    if f_at_10m * f_at_100m > 0:
        print("WARNING: d_max cannot be found in range 10 m – 100 m.") # założenie, że rozpatrujemy komunikację od 10 m do 100 m

    else:
        d_max = brentq(equation_to_solve_water, 10, 100)
        print(f"Maximum underwater comm distance: d_max = {d_max:.2f} m")

Maximum comm distance: d_max = 94.4 m  (0.094 km)


Obliczenie NEP, SNR i BER

In [ ]:
snr_nep = input("Calculate the NEP, SNR and BER values? ('YES' or 'NO'): ").strip().upper()

if snr_nep == "YES":
    A_D_mm2 = float(input("Enter detector area A_D (in mm^2): "))
    D_stand = float(input("Enter specific detectivity D* (in cm*sqrt(Hz)/W, e.g. 1e12): ")) # wykrywalność znormalizowana (z gwiazdką)
    B_MHz = float(input("Enter signal bandwidth B (in MHz): ")) # szerokość pasma, oznaczana jako (delta)f lub B

    A_D_cm2 = A_D_mm2 * 0.01 # konwertowanie jednostek, żeby były odpowiednie do wzoru
    B_Hz = B_MHz * 1e6

    NEP = math.sqrt(A_D_cm2 * B_Hz)/D_stand

    P_R_W = 10**(P_received_dBm / 10) / 1000

    SNR = (P_R_W / NEP)**2
    SNR_dB = 10 * math.log10(SNR) # konwersja na dB

    # BER - wzór zależy od wybranej wyżej modulacji
    if modulation == "OOK":
        BER = BER_OOK(SNR)
    elif modulation == "BPSK":
        BER = BER_BPSK(SNR)
    elif modulation == "PPM":
        BER = BER_PPM(SNR, M_order)
    elif modulation == "PAM":
        BER = BER_PAM(SNR, M_order)

    print(f"\nNEP / SNR / BER Results")
    print(f"NEP: {NEP:.3e} W  ({NEP*1e9:.4f} nW)")
    print(f"P_received: {P_R_W:.3e} W")
    print(f"SNR: {SNR:.2e}  ({SNR_dB:.1f} dB)")
    print(f"BER ({modulation}): {BER:.3e}")

    if SNR > 1:
        print("Signal detectable (SNR > 1)")
    else:
        print("Signal NOT detectable (SNR < 1)")


NEP / SNR / BER Results
NEP: 1.000e-09 W  (1.0000 nW)
P_received: 4.397e-06 W
SNR: 1.93e+07  (72.9 dB)
BER (OOK): 0.000e+00
Signal detectable (SNR > 1)
